In [14]:
# ===============================
# IMPORT LIBRARIES
# ===============================

import torch
import torch.nn as nn
import random


In [15]:
# ===============================
# LOAD DATASET
# ===============================

with open("TrainingNames.txt", "r") as f:
    names_data = f.read().splitlines()

print("Total names:", len(names_data))
print("Sample names:", names_data[:10])


Total names: 1000
Sample names: ['Aabharan', 'Aabharen', 'Aabharon', 'Aabheer', 'Aacharya', 'Aachman', 'Aadarsh', 'Aadav', 'Aadavan', 'Aadea']


In [16]:
# ===============================
# BUILD CHARACTER VOCABULARY
# ===============================

all_chars = sorted(list(set("".join(names_data))))
all_chars.append("<END>")

char_to_idx = {c:i for i,c in enumerate(all_chars)}
idx_to_char = {i:c for c,i in char_to_idx.items()}

vocab_size = len(all_chars)

print("Vocabulary size:", vocab_size)
print("Characters:", all_chars)


Vocabulary size: 50
Characters: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z', '<END>']


In [17]:
# ===============================
# ENCODING FUNCTION
# ===============================

def encode_name(name):
    return [char_to_idx[c] for c in name] + [char_to_idx["<END>"]]


In [18]:
# ===============================
# VANILLA RNN MODEL
# ===============================

class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.rnn = nn.RNN(hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out)
        return out


In [19]:

# ===============================
# BIDIRECTIONAL LSTM MODEL
# ===============================

class BLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, hidden_dim)

        self.lstm = nn.LSTM(
            hidden_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True,
            num_layers=2,      # 🔥 IMPORTANT FIX
            dropout=0.2
        )

        self.fc = nn.Linear(hidden_dim * 2, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = self.fc(out)
        return out

In [20]:

# ===============================
# TRAINING FUNCTION
# ===============================

def train_model(model, names, epochs=15):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # lower LR
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total_loss = 0

        for name in names:
            encoded = encode_name(name)

            inp = torch.tensor(encoded[:-1]).unsqueeze(0)
            target = torch.tensor(encoded[1:]).unsqueeze(0)

            output = model(inp)

            loss = loss_fn(output.view(-1, vocab_size),
                           target.view(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


In [21]:

# ===============================
# NAME GENERATION
# ===============================

def generate_name(model, max_len=12):
    model.eval()

    # Start with uppercase only
    start_chars = [c for c in char_to_idx if c.isupper()]
    char = random.choice(start_chars)

    name = char

    for _ in range(max_len):
        inp = torch.tensor([[char_to_idx[char]]])
        out = model(inp)

        probs = torch.softmax(out[0][-1], dim=0)

        next_idx = torch.multinomial(probs, 1).item()
        char = idx_to_char[next_idx]

        # Skip invalid tokens
        if char == "<END>":
            break

        if not char.isalpha():   # remove junk
            continue

        name += char

    return name


In [22]:

# ===============================
# EVALUATION METRICS
# ===============================

def novelty(generated, original):
    return len([n for n in generated if n not in original]) / len(generated)

def diversity(generated):
    return len(set(generated)) / len(generated)


In [28]:

# ===============================
# RUN EXAMPLE (RNN)
# ===============================

model = VanillaRNN(vocab_size, 64)
train_model(model, names_data, epochs=10)

generated = [generate_name(model) for _ in range(20)]
print("Generated Names:", generated)

print("Novelty:", novelty(generated, names_data))
print("Diversity:", diversity(generated))


Epoch 1, Loss: 2264.4473
Epoch 2, Loss: 1942.7074
Epoch 3, Loss: 1858.3328
Epoch 4, Loss: 1803.5023
Epoch 5, Loss: 1754.5479
Epoch 6, Loss: 1712.8473
Epoch 7, Loss: 1678.2906
Epoch 8, Loss: 1646.2912
Epoch 9, Loss: 1614.2553
Epoch 10, Loss: 1585.5464
Generated Names: ['Aanitmalanuks', 'OnWrihithraj', 'Vitikjoshallo', '<END>anjwthabezav', 'Trejananjilil', 'HathajoChishy', 'Galanguthabhu', 'Jai', 'Yathathishani', 'Ma', 'Wrikejishadhe', 'Vishakriludhi', 'FapagrilXdanc', 'Yajithismulak', 'Ujndilpthilru', 'Lalivangmkhri', 'Yandikanemaka', 'Ranshikshrind', 'Ekatrillanagt', 'Laluddevehuta']
Novelty: 0.95
Diversity: 1.0


In [29]:
import torch

total_params = sum(p.numel() for p in model.parameters())
model_size_mb = total_params * 4 / (1024 * 1024)

print("Total Parameters:", total_params)
print("Model Size (MB):", round(model_size_mb, 4))

Total Parameters: 14770
Model Size (MB): 0.0563


In [24]:
# ===============================
# TRAIN BLSTM MODEL
# ===============================

blstm_model = BLSTM(vocab_size, 64)

print("\nTraining BLSTM...\n")
train_model(blstm_model, names_data, epochs=10)

# Generate names
blstm_generated = [generate_name(blstm_model) for _ in range(30)]

print("\nBLSTM Generated Names:")
print(blstm_generated)

# Evaluate
print("\nBLSTM Novelty:", novelty(blstm_generated, names_data))
print("BLSTM Diversity:", diversity(blstm_generated))


Training BLSTM...

Epoch 1, Loss: 1038.1878
Epoch 2, Loss: 112.1569
Epoch 3, Loss: 27.8634
Epoch 4, Loss: 10.3530
Epoch 5, Loss: 5.7159
Epoch 6, Loss: 3.2568
Epoch 7, Loss: 1.6182
Epoch 8, Loss: 0.9634
Epoch 9, Loss: 0.4937
Epoch 10, Loss: 4.2162

BLSTM Generated Names:
['Ka', 'J', 'Y', 'V', 'D', 'N', 'J', 'O', 'R', 'F', 'B', 'P', 'P', 'T', 'D', 'N', 'F', 'D', 'W', 'Z', 'B', 'Z', 'R', 'F', 'W', 'J', 'C', 'X', 'W', 'C']

BLSTM Novelty: 1.0
BLSTM Diversity: 0.5333333333333333


In [25]:
# ===============
# ATTENTION MODEL
# ===============

import torch
import torch.nn as nn

class AttentionRNN(nn.Module):
    def __init__(self, vocab_size, hidden_dim):
        super().__init__()

        # Embedding layer: converts character index → vector
        self.embedding_layer = nn.Embedding(vocab_size, hidden_dim)

        # RNN layer: processes sequence
        self.sequence_model = nn.RNN(hidden_dim, hidden_dim, batch_first=True)

        # Attention layer: learns importance of each timestep
        self.attention_layer = nn.Linear(hidden_dim, hidden_dim)

        # Output layer: maps hidden state → vocabulary
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_seq):

        # Step 1: Convert input indices to embeddings
        embedded_seq = self.embedding_layer(input_seq)

        # Step 2: Pass through RNN
        rnn_outputs, _ = self.sequence_model(embedded_seq)
        # Shape: [batch_size, seq_len, hidden_dim]

        # Step 3: Compute attention weights
        attention_scores = self.attention_layer(rnn_outputs)
        attention_weights = torch.softmax(attention_scores, dim=2)

        # Step 4: Apply attention
        attended_output = rnn_outputs * attention_weights

        # Step 5: Final output (sequence-wise)
        final_logits = self.output_layer(attended_output)
        # Shape: [batch_size, seq_len, vocab_size]

        return final_logits


# ===============================
# TRAIN + TEST ATTENTION MODEL
# ===============================

print("===== ATTENTION MODEL =====")

attention_model = AttentionRNN(vocab_size, 64)

# Train model
train_model(attention_model, names_data, epochs=10)

# Generate names
generated_attention = [generate_name(attention_model) for _ in range(30)]

print("\nGenerated Names (Attention):")
print(generated_attention)

# Evaluation
print("\nNovelty:", novelty(generated_attention, names_data))
print("Diversity:", diversity(generated_attention))

===== ATTENTION MODEL =====
Epoch 1, Loss: 3172.1998
Epoch 2, Loss: 2726.2459
Epoch 3, Loss: 2658.4113
Epoch 4, Loss: 2636.8722
Epoch 5, Loss: 2625.0623
Epoch 6, Loss: 2606.7532
Epoch 7, Loss: 2523.8158
Epoch 8, Loss: 2331.4789
Epoch 9, Loss: 2225.5385
Epoch 10, Loss: 2131.2539

Generated Names (Attention):
['TavesK', 'Pt', 'Zidseri', 'Yharavk', 'Kimeejiialaad', 'Fashai', 'Endhi', 'Bamacdhyagaoj', 'Curan', 'Kadirhaludhad', 'Leykhthaymevk', 'Irdihajain', 'Shueemadijara', 'Fanseiidvvimy', 'Rariramijarav', 'Nrshinh', 'Peksemajacani', 'Jarisimaraln', 'Zshanlitjavas', 'Mavehinra', 'Gennastktk', 'Fijiyahoyienh', 'Niujaculas', 'Iimyashiisnhe', 'Mnliipabejiua', 'Iaj', 'Larama', 'Cwrasyauduset', 'Wdi', 'Hi']

Novelty: 1.0
Diversity: 1.0
